In [1]:
import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Activation
from music21 import converter, instrument, note, chord

In [2]:
def load_midi_notes(folder_path):
    notes = []
    print("Reading MIDI files...")
    
    # Look for all .mid files in the specified directory
    for file in glob.glob(os.path.join(folder_path, "*.mid")):
        print(f"Parsing: {file}")
        try:
            midi = converter.parse(file)
            # Try to isolate individual instrument parts
            parts = instrument.partitionByInstrument(midi)
            if parts: 
                notes_to_parse = parts.parts[0].recurse()
            else: 
                notes_to_parse = midi.flat.notes

            for element in notes_to_parse:
                if isinstance(element, note.Note):
                    notes.append(str(element.pitch))
                elif isinstance(element, chord.Chord):
                    # Chords are collections of notes; string them together with dots
                    notes.append('.'.join(str(n) for n in element.normalOrder))
        except Exception as e:
            print(f"Failed to read {file}: {e}")
            
    return notes

In [3]:
# Load the data (Make sure this folder exists and has midi files!)
path_to_midis = "midi_dataset"
if not os.path.exists(path_to_midis):
    os.makedirs(path_to_midis)
    print(f"Created '{path_to_midis}' directory. Please drop some .mid files inside and rerun.")
    exit()

all_notes = load_midi_notes(path_to_midis)

if len(all_notes) == 0:
    print("No notes found. Please put MIDI files in the folder.")
    exit()

Reading MIDI files...
Parsing: midi_dataset\x (1).mid
Parsing: midi_dataset\x (10).mid
Parsing: midi_dataset\x (11).mid
Parsing: midi_dataset\x (12).mid
Parsing: midi_dataset\x (13).mid
Parsing: midi_dataset\x (14).mid
Parsing: midi_dataset\x (15).mid
Parsing: midi_dataset\x (16).mid
Parsing: midi_dataset\x (17).mid
Parsing: midi_dataset\x (18).mid
Parsing: midi_dataset\x (19).mid
Parsing: midi_dataset\x (2).mid
Parsing: midi_dataset\x (20).mid
Parsing: midi_dataset\x (21).mid
Parsing: midi_dataset\x (22).mid
Parsing: midi_dataset\x (23).mid
Parsing: midi_dataset\x (24).mid
Parsing: midi_dataset\x (25).mid
Parsing: midi_dataset\x (26).mid
Parsing: midi_dataset\x (27).mid
Parsing: midi_dataset\x (28).mid
Parsing: midi_dataset\x (29).mid
Parsing: midi_dataset\x (3).mid


c:\Users\SM Rifat\Desktop\Project_for_Data\Project Ai\venv\Lib\site-packages\music21\midi\translate.py:1993: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=8, data=b'Shawn Mendes, Camila Cabello \x96 Se\xf1orita'>; getting generic Instrument
  metaObj = midiEventToInstrument(e, encoding=encoding)


Parsing: midi_dataset\x (30).mid
Parsing: midi_dataset\x (31).mid
Parsing: midi_dataset\x (32).mid
Parsing: midi_dataset\x (33).mid
Parsing: midi_dataset\x (34).mid
Parsing: midi_dataset\x (35).mid
Parsing: midi_dataset\x (36).mid
Parsing: midi_dataset\x (37).mid
Parsing: midi_dataset\x (38).mid
Parsing: midi_dataset\x (39).mid
Parsing: midi_dataset\x (4).mid
Parsing: midi_dataset\x (40).mid
Parsing: midi_dataset\x (41).mid
Parsing: midi_dataset\x (42).mid
Parsing: midi_dataset\x (43).mid
Parsing: midi_dataset\x (44).mid
Parsing: midi_dataset\x (45).mid
Parsing: midi_dataset\x (46).mid
Parsing: midi_dataset\x (47).mid
Parsing: midi_dataset\x (48).mid
Parsing: midi_dataset\x (49).mid
Parsing: midi_dataset\x (5).mid
Parsing: midi_dataset\x (50).mid
Parsing: midi_dataset\x (6).mid
Parsing: midi_dataset\x (7).mid


c:\Users\SM Rifat\Desktop\Project_for_Data\Project Ai\venv\Lib\site-packages\music21\midi\translate.py:1993: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=7, data=b'The Chainsmokers feat. XYL\xd8 - Setting Fires'>; getting generic Instrument
  metaObj = midiEventToInstrument(e, encoding=encoding)


Parsing: midi_dataset\x (8).mid
Parsing: midi_dataset\x (9).mid


In [4]:
# Get all unique notes (The trick we learned earlier!)
unique_notes = sorted(list(set(all_notes)))
n_vocab = len(unique_notes)
print(f"Total notes parsed: {len(all_notes)}")
print(f"Unique notes vocabulary size: {n_vocab}")

# Create a dictionary to map musical string notes to integers
note_to_int = {note: num for num, note in enumerate(unique_notes)}
int_to_note = {num: note for num, note in enumerate(unique_notes)}

# =====================================================================
# 2. PREPARING SEQUENCES FOR LSTM
# =====================================================================
sequence_length = 100  # Look at 100 notes to predict the 101st
network_input = []
network_output = []

for i in range(0, len(all_notes) - sequence_length):
    seq_in = all_notes[i:i + sequence_length]
    seq_out = all_notes[i + sequence_length]
    
    network_input.append([note_to_int[char] for char in seq_in])
    network_output.append(note_to_int[seq_out])

n_patterns = len(network_input)

Total notes parsed: 3892
Unique notes vocabulary size: 113


In [5]:
# Reshape input to 3D: [samples, time_steps, features]
X_train = np.reshape(network_input, (n_patterns, sequence_length, 1))
# Normalize the inputs between 0 and 1
X_train = X_train / float(n_vocab)

# One-hot encode the output targets
y_train = tf.keras.utils.to_categorical(network_output, num_classes=n_vocab)

# =====================================================================
# 3. BUILD THE LSTM DEEP LEARNING MODEL
# =====================================================================
model = Sequential([
    LSTM(256, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True),
    Dropout(0.3),
    LSTM(256),
    Dropout(0.3),
    Dense(n_vocab),
    Activation('softmax') # Outputs probabilities for each unique note
])

model.compile(loss='categorical_crossentropy', optimizer='rmsprop')

# =====================================================================
# 4. TRAINING THE MODEL
# =====================================================================
print("\nTraining starting... (Adjust epochs based on your patience/hardware)")
# Real training takes 50+ epochs. We set 5 here just to prove the code works.
model.fit(X_train, y_train, epochs=5, batch_size=64) 

# =====================================================================
# 5. MUSIC GENERATION
# =====================================================================
print("\nGenerating new music...")
# Pick a random starting sequence from our data to kick off the generation
start_index = np.random.randint(0, len(network_input)-1)
pattern = network_input[start_index]
prediction_output = []

c:\Users\SM Rifat\Desktop\Project_for_Data\Project Ai\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Training starting... (Adjust epochs based on your patience/hardware)
Epoch 1/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - loss: 4.2632
Epoch 2/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - loss: 4.1404
Epoch 3/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - loss: 3.9910
Epoch 4/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - loss: 3.8694
Epoch 5/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - loss: 3.8028

Generating new music...


In [6]:
# Generate 200 notes
for note_index in range(200):
    prediction_input = np.reshape(pattern, (1, len(pattern), 1))
    prediction_input = prediction_input / float(n_vocab)
    
    prediction = model.predict(prediction_input, verbose=0)
    index = np.argmax(prediction) # Pick the note with the highest probability
    
    result = int_to_note[index]
    prediction_output.append(result)
    
    pattern.append(index)
    pattern = pattern[1:len(pattern)] # Slide the window forward by 1 note

# =====================================================================
# 6. CONVERT GENERATED SEQUENCES BACK TO MIDI FILE
# =====================================================================
offset = 0
output_notes = []

# Recreate note and chord objects from strings
for pattern in prediction_output:
    # If the pattern is a chord string (contains a dot)
    if ('.' in pattern) or pattern.isdigit():
        notes_in_chord = pattern.split('.')
        notes = []
        for current_note in notes_in_chord:
            new_note = note.Note(int(current_note))
            new_note.storedInstrument = instrument.Piano()
            notes.append(new_note)
        new_chord = chord.Chord(notes)
        new_chord.offset = offset
        output_notes.append(new_chord)
    # If the pattern is a single note string
    else:
        new_note = note.Note(pattern)
        new_note.offset = offset
        new_note.storedInstrument = instrument.Piano()
        output_notes.append(new_note)
        
    # Move the timestamp forward slightly so notes don't stack on top of each other
    offset += 0.5

# Save to file
midi_stream = converter.stream.Stream(output_notes)
output_filename = 'ai_generated_composition.mid'
midi_stream.write('midi', fp=output_filename)
print(f"\nSuccess! Saved generated music file to: {os.path.abspath(output_filename)}")


Success! Saved generated music file to: c:\Users\SM Rifat\Desktop\Project_for_Data\Project Ai\Music_Generation\ai_generated_composition.mid
